In [ ]:
"""
Live viewer that reads from the .npz cache written by sliced_poller.py.
Never touches the openPMD series directly — safe while WarpX is running.
"""

import json, os, threading, time
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import ipywidgets as widgets
from IPython.display import display

%matplotlib widget

CACHE_DIR = "./diags/fields_sliced_cache/"
MANIFEST  = os.path.join(CACHE_DIR, "manifest.json")
VMIN, VMAX    = -1e5, 1e5
SLICE_AX      = 0        # WarpX writes (Nz, Ny, Nx) — axis 0 is the thin Z slab

_eb_cmap = mcolors.LinearSegmentedColormap.from_list(
    'eb_mask', [(0,0,0,0), (0,0,0,1)])

# ── Cache (populated by poll thread) ─────────────────────────────────────────
iter_list  = []   # sorted iteration indices (ints)
iter_data  = {}   # index -> {"arr": ndarray, "extent": list, "t": float}
_lock      = threading.Lock()
_stop      = threading.Event()


def _load_manifest():
    """Return manifest dict or {} if not yet available."""
    try:
        with open(MANIFEST) as f:
            return json.load(f)
    except Exception:
        return {}


def _poll():
    known = set()
    while not _stop.is_set():
        mf = _load_manifest()
        for k, v in mf.items():
            idx = int(k)
            if idx in known:
                continue
            try:
                d = np.load(v["file"])
                with _lock:
                    iter_data[idx] = {
                        "arr": d["arr"],
                        "extent": list(d["extent"]),
                        "t": float(d["t"]),
                    }
                    if idx not in iter_list:
                        iter_list.append(idx)
                        iter_list.sort()
                known.add(idx)
            except Exception:
                pass   # file may still be mid-write
        _stop.wait(POLL_INTERVAL)


_stop.clear()
threading.Thread(target=_poll, daemon=True).start()

# Wait for at least one iteration
print("Waiting for first cached iteration", end="", flush=True)
while not iter_list:
    time.sleep(0.5)
    print(".", end="", flush=True)
print(f" got iteration {iter_list[0]}.")

# ── Load eb_covered if available ──────────────────────────────────────────────
eb_path = os.path.join(CACHE_DIR, "eb_covered.npy")
eb_arr  = np.load(eb_path) if os.path.exists(eb_path) else None

# ── Build figure once ─────────────────────────────────────────────────────────
with _lock:
    i0 = iter_list[0]
    d0 = iter_data[i0]

ext0  = d0["extent"]
iext0 = [ext0[2], ext0[3], ext0[0], ext0[1]]  # [ymin,ymax,xmin,xmax] for imshow

fig, ax = plt.subplots(figsize=(10, 6))
fig.canvas.header_visible = False

im_f = ax.imshow(d0["arr"], extent=iext0, origin='lower',
                 aspect='equal', cmap='RdBu', vmin=VMIN, vmax=VMAX)
plt.colorbar(im_f, ax=ax, label="Ez (V/m)")

im_eb = None
if eb_arr is not None:
    im_eb = ax.imshow(eb_arr, extent=iext0, origin='lower',
                      aspect='equal', cmap=_eb_cmap, vmin=0, vmax=1,
                      interpolation='nearest')

ttl = ax.set_title(f"Iteration {i0}  t = {d0['t']*1e9:.4f} ns")
ax.set_xlabel("y (m)"); ax.set_ylabel("x (m)")
fig.tight_layout()

# ── Widgets ───────────────────────────────────────────────────────────────────
slider = widgets.IntSlider(
    min=0, max=0, value=0, step=1,
    description="Iteration:",
    continuous_update=False,
    layout=widgets.Layout(width="600px"),
    style={"description_width": "80px"},
)
play   = widgets.Play(min=0, max=0, step=1, interval=200)
widgets.jslink((play, "value"), (slider, "value"))
status = widgets.Label(value=f"Cached: {len(iter_list)} iterations")
btn    = widgets.Button(description="↺ Refresh", button_style="info")


def _render(pos):
    with _lock:
        if pos >= len(iter_list):
            return
        i = iter_list[pos]
        d = iter_data[i]
    im_f.set_data(d["arr"])
    if im_eb is not None and eb_arr is not None:
        im_eb.set_data(eb_arr)
    ttl.set_text(f"Iteration {i}  t = {d['t']*1e9:.4f} ns")
    fig.canvas.draw_idle()


def _refresh(_=None):
    with _lock:
        n = len(iter_list)
        last = iter_list[-1] if iter_list else 0
    slider.max = max(0, n - 1)
    play.max   = max(0, n - 1)
    status.value = f"Cached: {n} iterations  (last: {last})"


slider.observe(lambda c: _render(c["new"]), names="value")
btn.on_click(_refresh)

# Auto-refresh label in background
def _auto():
    while not _stop.is_set():
        _stop.wait(POLL_INTERVAL)
        _refresh()
threading.Thread(target=_auto, daemon=True).start()

display(widgets.VBox([
    widgets.HBox([play, slider]),
    widgets.HBox([btn, status]),
]))
_render(0)
print("To stop polling: _stop.set()")